In [0]:
dbutils.widgets.removeAll()

In [0]:
pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)

from pyspark.sql.functions import col, current_timestamp

from openpyxl import load_workbook
from io import BytesIO
from datetime import datetime
import re

In [0]:
dbutils.widgets.text("container", "exlt-raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "stccobankanticsdevcaeast")
dbutils.widgets.text("tabla", "sfc_indicadores_raw")

In [0]:
%sql

SHOW EXTERNAL LOCATIONS;

In [0]:
%sql

CREATE EXTERNAL VOLUME IF NOT EXISTS
catalog_au.bronze.vol_sfc_raw
LOCATION
'abfss://exlt-raw@stccobankanticsdevcaeast.dfs.core.windows.net/'
COMMENT 'Volumen externo para archivos Excel originales de la SFC';

In [0]:
%sql

DESCRIBE VOLUME catalog_au.bronze.vol_sfc_raw;

In [0]:
# Parámetros de las celdas anteriores
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
tabla = dbutils.widgets.get("tabla")

# Ruta administrada por Unity Catalog
ruta = f"/Volumes/{catalogo}/{esquema}/vol_sfc_raw/"

tabla_destino = f"{catalogo}.{esquema}.{tabla}"

print("Ruta origen:", ruta)
print("Tabla destino:", tabla_destino)

# Listar archivos Excel en la raíz del volumen
archivos = [
    archivo
    for archivo in dbutils.fs.ls(ruta)
    if archivo.name.lower().endswith(".xlsx")
    and not archivo.name.startswith("~$")
]

print(f"Archivos Excel encontrados: {len(archivos)}")

for archivo in archivos:
    print(archivo.name, archivo.size, "bytes")

In [0]:
from openpyxl import load_workbook
from io import BytesIO
from datetime import datetime
import re

def extraer_anio(nombre_archivo):
    coincidencia = re.search(
        r"ig_(\d{4})_(\d{2})\.xlsx$",
        nombre_archivo,
        re.IGNORECASE
    )
    return int(coincidencia.group(1)) if coincidencia else None


def leer_excel(ruta_archivo):

    nombre_archivo = ruta_archivo.rsplit("/", 1)[-1]
    anio = extraer_anio(nombre_archivo)

    # Leer el Excel desde el volumen de Unity Catalog
    with open(ruta_archivo, "rb") as archivo:
        contenido = archivo.read()

    workbook = load_workbook(
        BytesIO(contenido),
        read_only=True,
        data_only=True
    )

    registros = []

    try:
        for hoja in workbook.worksheets:

            for fila in hoja.iter_rows():

                for celda in fila:

                    if celda.value is None:
                        continue

                    valor = celda.value

                    if isinstance(valor, datetime):
                        valor_texto = valor.isoformat()
                    else:
                        valor_texto = str(valor)

                    registros.append((
                        nombre_archivo,
                        anio,
                        hoja.title,
                        celda.row,
                        celda.column,
                        celda.coordinate,
                        valor_texto,
                        celda.data_type,
                        None
                    ))

    finally:
        workbook.close()

    return registros

In [0]:
# Seleccionar archivo de enero de 2024
archivo_prueba = next(
    archivo
    for archivo in archivos
    if archivo.name == "ig_2024_01.xlsx"
)

# Convertir ruta dbfs:/Volumes a /Volumes
ruta_archivo = archivo_prueba.path.replace(
    "dbfs:/Volumes/",
    "/Volumes/"
)

print("Ruta original:", archivo_prueba.path)
print("Ruta corregida:", ruta_archivo)

# Leer el Excel
registros_prueba = leer_excel(ruta_archivo)

print("Archivo:", archivo_prueba.name)
print("Total de celdas:", len(registros_prueba))

for registro in registros_prueba[:10]:
    print(registro)

In [0]:
# CELDA 5 - Definición del esquema Bronze

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)

sfc_schema = StructType([
    StructField("archivo_origen", StringType(), True),
    StructField("anio_origen", IntegerType(), True),
    StructField("hoja", StringType(), True),
    StructField("fila_excel", IntegerType(), True),
    StructField("columna_excel", IntegerType(), True),
    StructField("coordenada_excel", StringType(), True),
    StructField("valor_texto", StringType(), True),
    StructField("tipo_valor", StringType(), True),
    StructField("fecha_ingesta", TimestampType(), True)
])

print("Esquema Bronze definido correctamente")

In [0]:
# Construir el DataFrame con los registros ya leídos

from pyspark.sql.functions import current_timestamp

df_sfc = (
    spark.createDataFrame(
        registros,
        schema=sfc_schema
    )
    .withColumn(
        "fecha_ingesta",
        current_timestamp()
    )
)

print("Total de registros:", df_sfc.count())

df_sfc.printSchema()

display(df_sfc.limit(20))

In [0]:
# CELDA 8 - Ingesta a Delta Lake Bronze

tabla_destino = "catalog_au.bronze.sfc_indicadores_raw"

# Comprobar que los archivos no estén cargados
archivos_nuevos = [
    fila["archivo_origen"]
    for fila in df_sfc.select("archivo_origen").distinct().collect()
]

archivos_existentes = {
    fila["archivo_origen"]
    for fila in (
        spark.table(tabla_destino)
        .select("archivo_origen")
        .distinct()
        .collect()
    )
}

duplicados = set(archivos_nuevos) & archivos_existentes

if duplicados:
    raise ValueError(
        f"Los siguientes archivos ya existen en Bronze: {duplicados}. "
        "Se cancela la carga para evitar duplicados."
    )

# Insertar registros
(
    df_sfc.write
    .format("delta")
    .mode("append")
    .saveAsTable(tabla_destino)
)

print("Ingesta Bronze completada correctamente")
print("Tabla destino:", tabla_destino)

In [0]:
%sql

SELECT
    archivo_origen,
    anio_origen,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT hoja) AS total_hojas,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM catalog_au.bronze.sfc_indicadores_raw
GROUP BY
    archivo_origen,
    anio_origen
ORDER BY
    anio_origen;